Install Required Libraries

In [1]:
%%capture
%pip install -U transformers datasets accelerate peft trl==0.12.2 bitsandbytes wandb


Login to Hugging Face & Weights and Biases

In [2]:
import os
from google.colab import userdata
from huggingface_hub import login
import wandb
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Set Runtime -> Change runtime type -> T4 GPU, then restart runtime.")

hf_token = userdata.get("HF_TOKEN")
wandb_token = userdata.get("WANDB_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN missing in Colab Secrets or notebook access not enabled.")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)
print("Hugging Face login successful.")

use_wandb = bool(wandb_token)
if use_wandb:
    os.environ["WANDB_API_KEY"] = wandb_token
    wandb.login(key=wandb_token)
    run = wandb.init(
        project="Fine-tune-Qwen2.5-1.5B-on-Instruction-Dataset",
        job_type="training",
        anonymous="allow"
    )
    print("W&B login successful.")
else:
    print("WANDB_TOKEN not found. Training will continue without W&B logging.")


CUDA available: True


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vanshika-mehlawat-23cse (vanshika-mehlawat-23cse-bml-munjal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


W&B login successful.


 Import All Libraries

In [3]:
import re
import torch
from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

from trl import SFTTrainer, setup_chat_format


In [4]:
import pandas as pd
from datasets import Dataset, load_dataset

# 1. Load the dataset in streaming mode
print("Loading PubMed dataset stream...")
raw_dataset = load_dataset("ccdv/pubmed-summarization", "section", split="train", streaming=True)

# 2. Extract 1000 samples
print("Extracting 1000 samples...")
samples = []
iterator = iter(raw_dataset)
for _ in range(1000):
    try:
        samples.append(next(iterator))
    except StopIteration:
        break

# 3. Define the Prompt Template (Optimized for Qwen2.5)
def format_instruction(example):
    system_prompt = (
        "You are a medical assistant specializing in research summarization. "
        "Provide a concise, accurate, and professional summary of the medical text provided below."
    )
    # Formatting for SFTTrainer
    # Prompt is the input, completion is the target summary
    prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\nSummarize this medical text: {example['article']}<|im_end|>\n<|im_start|>assistant\n"
    completion = f"{example['abstract']}<|im_end|>"

    return {
        "prompt": prompt,
        "completion": completion,
        "full_text": prompt + completion # Useful for training
    }

# 4. Process and Split
formatted_data = [format_instruction(s) for s in samples]
dataset = Dataset.from_list(formatted_data)
dataset_split = dataset.train_test_split(test_size=200, seed=42) # 800 train, 200 test

train_dataset = dataset_split["train"]
test_dataset = dataset_split["test"]

# 5. Save the formatted data locally for documentation (Deliverable)
dataset_df = pd.DataFrame(formatted_data)
dataset_df.to_csv("formatted_pubmed_dataset.csv", index=False)

print(f"Dataset Prepared!")
print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(test_dataset)}")
print("File 'formatted_pubmed_dataset.csv' has been saved for your assignment submission.")

Loading PubMed dataset stream...


README.md: 0.00B [00:00, ?B/s]

Extracting 1000 samples...
Dataset Prepared!
Training samples: 800
Evaluation samples: 200
File 'formatted_pubmed_dataset.csv' has been saved for your assignment submission.


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # T4 supports bfloat16 for computation
)

# 2. Load Tokenizer
print(f"Loading tokenizer: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Recommended for SFT

# 3. Load Model
print(f"Loading model: {model_id}")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 4. Prepare model for k-bit training (Gradient Checkpointing, etc.)
model = prepare_model_for_kbit_training(model)

# 5. Configure LoRA (Low-Rank Adaptation)
# We target all linear layers to get the best performance for medical text
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 6. Apply LoRA to the model
model = get_peft_model(model, lora_config)

# 7. Statistics
print("\n--- Model Ready ---")
model.print_trainable_parameters()

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading model: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


--- Model Ready ---
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen-pubmed-finetuned",
    per_device_train_batch_size=2,      # Small batch size for T4 GPU
    gradient_accumulation_steps=4,     # Effective batch size = 2 * 4 = 8
    optim="paged_adamw_32bit",         # Memory efficient optimizer
    learning_rate=2e-4,                # Standard for LoRA
    lr_scheduler_type="cosine",
    save_strategy="epoch",             # Save a checkpoint every epoch
    logging_steps=10,                  # Log to W&B every 10 steps
    num_train_epochs=3,                # 3 epochs for good performance
    max_steps=-1,                      # Use epochs instead of steps
    fp16=True,                         # Use mixed precision (essential for T4)
    group_by_length=True,              # Speed up training by grouping similar length texts
    report_to="wandb" if use_wandb else "none",
    run_name="qwen2.5-pubmed-summarization", # Name for your W&B run
)

# 2. Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=lora_config,
    dataset_text_field="full_text",    # The field we created in Step 1
    max_seq_length=1024,               # PubMed sections can be long
    tokenizer=tokenizer,
    args=training_args,
    packing=False,                     # Change to True if you want to speed up training further
)

# 3. Start Training
print("Starting training... Check your W&B dashboard for live graphs!")
trainer.train()

# 4. Save the Final Model Adapter
trainer.model.save_pretrained("./final_qwen_pubmed_adapter")
tokenizer.save_pretrained("./final_qwen_pubmed_adapter")

print("\nTraining Complete!")
print("The LoRA adapter has been saved to './final_qwen_pubmed_adapter'")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Starting training... Check your W&B dashboard for live graphs!


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.202800
20,2.043400
30,1.978600
40,1.984400
50,2.023100
60,2.035800
70,1.970800
80,1.995700
90,2.031600
100,1.954500


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



Training Complete!
The LoRA adapter has been saved to './final_qwen_pubmed_adapter'


In [8]:
%pip install evaluate rouge_score sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=66972a43ae8ae10e44ee031e3b2adea59745b6a3f571c193f54e129be257621d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [10]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory Cleared.")

GPU Memory Cleared.


In [12]:
import torch

# 1. Clear memory again just to be safe
import gc
gc.collect()
torch.cuda.empty_cache()

# 2. Pick the very first test sample
test_item = test_dataset[0]
test_prompt = test_item['prompt']

print("--- Testing Single Sample Generation ---")
print("Input Prompt Length:", len(test_prompt))

# 3. Try to generate ONE summary
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")

print("Sending to GPU... Please wait (this should take 15-30 seconds)...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50, # Very short for testing
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True # Speeds up generation
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- TEST SUCCESSFUL ---")
print("Generated Text Preview:", result.split("assistant")[-1][:100])

--- Testing Single Sample Generation ---
Input Prompt Length: 18786
Sending to GPU... Please wait (this should take 15-30 seconds)...

--- TEST SUCCESSFUL ---
Generated Text Preview:  specializing in research summarization. Provide a concise, accurate, and professional summary of th


In [14]:
import os
from google.colab import files

# Zip the final adapter folder
os.system("zip -r final_qwen_pubmed_adapter.zip ./final_qwen_pubmed_adapter")

# Download the zip
print("Downloading Final Adapter...")
files.download("final_qwen_pubmed_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Zip the training directory (contains all checkpoints)
os.system("zip -r training_checkpoints.zip ./qwen-pubmed-finetuned")

# Download the zip
print("Downloading Training Checkpoints (This may be large)...")
files.download("training_checkpoints.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
# List of all CSVs we created
csv_files = [f for f in os.listdir('.') if f.endswith('.csv') or f.endswith('.txt')]

# Zip all CSVs and documentation
import zipfile
with zipfile.ZipFile('all_datasets_and_docs.zip', 'w') as zipf:
    for file in csv_files:
        zipf.write(file)

print("Downloading all CSVs and Documentation...")
files.download("all_datasets_and_docs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import torch
import pandas as pd
from evaluate import load
import gc

# 1. Clear GPU memory to prevent another hang
gc.collect()
torch.cuda.empty_cache()

# 2. Load Metrics
print("Loading metrics libraries...")
rouge = load("rouge")
bleu = load("bleu")

# 3. Select only 5 samples for a quick result
num_samples = 5
eval_subset = test_dataset.select(range(num_samples))

base_preds, ft_preds, references = [], [], []

def quick_generate(model, tokenizer, text):
    # Truncate input to 1500 chars to ensure it's ultra-fast
    prompt = f"<|im_start|>system\nYou are a medical assistant.<|im_end|>\n<|im_start|>user\nSummarize this: {text[:1500]}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

# 4. Generate Side-by-Side
print(f"Generating 5 comparisons (Base vs Fine-Tuned)...")
for i in range(num_samples):
    article = eval_subset[i]['prompt'].split("medical text: ")[-1].split("<|im_end|>")[0]
    ref = eval_subset[i]['completion'].replace("<|im_end|>", "").strip()

    # Fine-Tuned
    ft_out = quick_generate(model, tokenizer, article)
    ft_preds.append(ft_out)

    # Base Model
    with model.disable_adapter():
        base_out = quick_generate(model, tokenizer, article)
        base_preds.append(base_out)

    references.append(ref)
    print(f"Sample {i+1}/5 Complete.")

# 5. Calculate Final Metrics
ft_rouge = rouge.compute(predictions=ft_preds, references=references)
base_rouge = rouge.compute(predictions=base_preds, references=references)
ft_bleu = bleu.compute(predictions=ft_preds, references=references)
base_bleu = bleu.compute(predictions=base_preds, references=references)

# 6. Manual Keyword Accuracy Score
def get_acc(p, r):
    scores = []
    for pred, ref in zip(p, r):
        common = set(pred.lower().split()) & set(ref.lower().split())
        scores.append(len(common) / len(set(ref.lower().split())) if ref else 0)
    return sum(scores) / len(scores)

ft_acc = get_acc(ft_preds, references)
base_acc = get_acc(base_preds, references)

# --- THE FINAL TABLE FOR YOUR REPORT ---
final_results = pd.DataFrame({
    "Metric": ["ROUGE-L", "BLEU", "Keyword Accuracy"],
    "Base Model": [base_rouge['rougeL'], base_bleu['bleu'], base_acc],
    "Fine-Tuned Model": [ft_rouge['rougeL'], ft_bleu['bleu'], ft_acc]
})

print("\n" + "="*40)
print("FINAL ASSIGNMENT RESULTS (N=5)")
print(final_results.to_string(index=False))
print("="*40)

# Save this for your report screenshot!
final_results.to_csv("Final_Report_Metrics.csv", index=False)

Loading metrics libraries...
Generating 5 comparisons (Base vs Fine-Tuned)...
Sample 1/5 Complete.
Sample 2/5 Complete.
Sample 3/5 Complete.
Sample 4/5 Complete.
Sample 5/5 Complete.

FINAL ASSIGNMENT RESULTS (N=5)
          Metric  Base Model  Fine-Tuned Model
         ROUGE-L    0.087035          0.060903
            BLEU    0.000000          0.001773
Keyword Accuracy    0.093749          0.113219


In [18]:
from rouge_score import rouge_scorer
import wandb

# 1. Save the Fine-Tuned Model
new_model_name = "qwen2.5-1.5b-pubmed-summarizer"
trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)
print(f"Model saved as: {new_model_name}")

# 2. Setup the ROUGE Scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

# 3. Define the Medical Response Function
def generate_medical_summary(medical_text):
    # We use the official chat template for Qwen2.5
    messages = [
        {"role": "system", "content": "You are a medical research assistant. Summarize the following section into a professional abstract."},
        {"role": "user", "content": medical_text}
    ]

    # Apply Chat Template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to("cuda")

    # Generate Output
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract only the assistant's response
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text.split("assistant")[-1].strip()

# 4. Test Query (A real Medical Section from your test set)
sample_idx = 0
test_medical_section = test_dataset[sample_idx]['prompt'].split("medical text: ")[-1].split("<|im_end|>")[0][:1000]
expected_abstract = test_dataset[sample_idx]['completion'].replace("<|im_end|>", "").strip()

# 5. Run Inference
bot_response = generate_medical_summary(test_medical_section)

# 6. Calculate Score for this specific response
scores = scorer.score(expected_abstract, bot_response)

print("\n" + "="*30)
print("--- FINAL SYSTEM CHECK ---")
print(f"Input Section: {test_medical_section[:150]}...")
print(f"\nAI Medical Summary: {bot_response}")
print(f"\nExpected Summary: {expected_abstract[:150]}...")
print("-" * 30)
print(f"ROUGE-1 Score: {scores['rouge1'].fmeasure:.4f}")
print(f"ROUGE-L Score: {scores['rougeL'].fmeasure:.4f}")
print("="*30)

# 7. Close Weights & Biases Run
if use_wandb:
    wandb.finish()
    print("W&B session closed successfully.")

Model saved as: qwen2.5-1.5b-pubmed-summarizer


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



--- FINAL SYSTEM CHECK ---
Input Section: intestinal helminths are among the most common and widespread of human infections , contributing to poor nutritional status , anemia and impaired grow...

AI Medical Summary: introductionintestinal helminths are among the most common and widespread of human infections , contributing to poor nutritional status , anemia and impaired growth   ( 1 ) . 
 intestinal helminthiases are also known to aggravate pre - existing anemia by decreasing appetite and thus food and iron intake ( 2 , 3 ) . 
 worldwide , anemia is an important reproductive health problem because of its association with adverse pregnancy outcome such as increased rates of maternal and perinatal mortality , premature delivery , low birth weight , etc ( 4) . 
 women in developing countries spend half of their reproductive lives pregnant and lactating and a high proportion of women in developing countries become anemic during this period . 
 women of reproductive age who

Expected Sum

train/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/grad_norm,▁▁▁▁▆▂▁▁▁▃▂▂▂▃▆▃▃▃▃▅▃▃▄▃▆▄▄▄▄█
train/learning_rate,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▆▅▅▅▅▅▅▅▄▃▃▄▃▃▃▄▃▄▄▂▂▂▂▂▂▃▁▁▁
total_flos,1.9195518504861696e+16
train/epoch,3
train/global_step,300
train/grad_norm,0.65277
train/learning_rate,0
train/loss,1.7579


W&B session closed successfully.


In [20]:
import evaluate
from rouge_score import rouge_scorer

# 1. Load the metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

# 2. Select exactly 3 samples from the test set
num_cases = 3
eval_samples = test_dataset.select(range(num_cases))
predictions = []
references = [ex["completion"].replace("<|im_end|>", "").strip() for ex in eval_samples]

print(f"--- GENERATING {num_cases} MEDICAL CASE COMPARISONS ---")

for i, example in enumerate(eval_samples):
    # Extract the medical article text from our training prompt
    # We take a reasonable chunk (1200 chars) to ensure context without OOM
    article_text = example['prompt'].split("Summarize this medical text: ")[-1].split("<|im_end|>")[0][:1200]

    # Structure as an Instruction-Following Query
    messages = [
        {"role": "system", "content": "You are a medical research assistant. Provide a professional abstract summary of the following text."},
        {"role": "user", "content": article_text}
    ]

    # Apply Chat Template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to("cuda")

    # Generate Response
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract AI response
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()
    predictions.append(generated_text)

    # Print the Test Case clearly for your report
    print(f"\n[Medical Case {i+1}]")
    print(f"Medical Input Section: {article_text[:300]}...")
    print(f"\nExpected Abstract: {references[i][:300]}...")
    print(f"\nFine-Tuned Model Summary: {generated_text}")
    print("-" * 50)

# 3. Compute Metrics for these 3 cases
rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(predictions=predictions, references=references)

print("\n" + "="*45)
print("--- FINAL SUMMARY METRICS (N=3) ---")
print(f"ROUGE-L Score: {rouge_results['rougeL']:.4f}")
print(f"BLEU Score: {bleu_results['bleu']:.4f}")
print("="*45)

# Save this output as a final deliverable text file
with open("Three_Sample_Comparison.txt", "w") as f:
    f.write(f"ROUGE-L: {rouge_results['rougeL']:.4f}\n")
    f.write(f"BLEU: {bleu_results['bleu']:.4f}\n")
    f.write("\nRefer to the printed console for side-by-side text.")

--- GENERATING 3 MEDICAL CASE COMPARISONS ---

[Medical Case 1]
Medical Input Section: intestinal helminths are among the most common and widespread of human infections , contributing to poor nutritional status , anemia and impaired growth  ( 1 ) . 
 intestinal helminthiases are also known to aggravate pre - existing anemia by decreasing appetite and thus food and iron intake ( 2 , 3 ...

Expected Abstract: backgroundthe incidence and hematological effects of helminth infection during pregnancy were investigated among pregnant women in isiala , mbano , southeast nigeria.methods:totally 282 pregnant women were enlisted for the study between october 2011 and september 2012 . 
 stool samples were examined...

Fine-Tuned Model Summary: introduction : intestinal helminths are among the most common and widespread of human infections , contributing to poor nutritional status , anemia and impaired growth   ( 1 ) . 
 intestinal helminthiases are also known to aggravate pre - existing anemia by 